# 05 — Next Activity Prediction

Multiclass classification: given a prefix of k events, predict the activity label of the (k+1)-th event.
Methodology follows Teinemaa et al. (2019): prefix-length bucketing × three encodings × LightGBM sweep, then full-trace RF/XGB/LGBM and DL models.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
import pm4py as pm
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score,
    top_k_accuracy_score, average_precision_score
)
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

## 2. Load & Clean

In [3]:
dtype_dict = {
    'hired': 'Int8', 'Rejected': 'Int8', 'CW': 'Int8', 'Evergreen': 'Int8',
    'Region': 'Int16', 'Country': 'Int16',
    'Job Family': 'Int16', 'Job Family Group': 'Int16',
}
df = pd.read_csv( 'data/'+
    'event_log_anonymized_melted.csv',
    low_memory=False,
    dtype=dtype_dict,
    parse_dates=['timestamp']
)
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}')
print(f'Cases: {df["Case_id"].nunique():,}')

Shape: (1133314, 17)
Cases: 279,458


## 3. Format for pm4py + Chronological 60/20/20 Split

In [4]:
df = pm.format_dataframe(
    df, case_id='Case_id', activity_key='Step', timestamp_key='timestamp'
)

case_start = df.groupby('case:concept:name')['time:timestamp'].min().sort_values()
n_cases      = len(case_start)
train_cutoff = case_start.iloc[int(n_cases * 0.60)]
val_cutoff   = case_start.iloc[int(n_cases * 0.80)]

train_cases = case_start[case_start <  train_cutoff].index
val_cases   = case_start[(case_start >= train_cutoff) & (case_start < val_cutoff)].index
test_cases  = case_start[case_start >= val_cutoff].index

print(f'Train: {len(train_cases):,} | Val: {len(val_cases):,} | Test: {len(test_cases):,}')
print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')

train_df = df[df['case:concept:name'].isin(train_cases)].copy()
val_df   = df[df['case:concept:name'].isin(val_cases)].copy()
test_df  = df[df['case:concept:name'].isin(test_cases)].copy()

Train: 167,674 | Val: 55,892 | Test: 55,892
Train cutoff: 2024-09-12 | Val cutoff: 2025-08-29


## 4. Remove Leaky Activities + Build Vocabulary

In [5]:
LEAKY_THRESHOLD = 0.85
act_hire_rate = (
    train_df.groupby('concept:name')['hired']
    .apply(lambda x: x.astype(float).mean())
    .sort_values(ascending=False)
)
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)
print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

# Activity vocabulary — fit on train only
all_train_acts = sorted(train_df['concept:name'].dropna().unique())
act2idx = {act: i+2 for i, act in enumerate(all_train_acts)}
act2idx['<PAD>'] = 0
act2idx['<UNK>'] = 1
idx2act = {v: k for k, v in act2idx.items()}
VOCAB_SIZE  = len(act2idx)
MAX_SEQ_LEN = 30
print(f'\nVocabulary size: {VOCAB_SIZE} | MAX_SEQ_LEN: {MAX_SEQ_LEN}')

Leaky activities removed (17):
  1.000  Ready for Hire – Do not send Manager Survey
  1.000  Ready for Hire - Send Hiring Manager Survey
  0.997  Additional Offer Packet Document(s)
  0.990  Review Company and Office Guides
  0.986  Candidate Experience Survey
  0.984  Recruiting Process Survey
  0.976  Automatically Unpost Jobs
  0.968  Confirm no manager survey to be sent
  0.912  Colombia Standard Questionnaire (2024)
  0.909  Additional Background / Reference Check
  0.909  LATAM Standard Questionnaire (2024)
  0.900  Confirm no manager/additional hiring manager survey to be sent
  0.889  Renegotiate Offer
  0.885  Make Background Check Decision
  0.885  BGC / Examenes Preocupacionales
  0.884  Set Background Check Status
  0.853  Consolidated Approval by Initiator

Vocabulary size: 39 | MAX_SEQ_LEN: 30


## 5. Encoding Helpers (Boolean / Frequency / Succession)

Identical to NB04. Fit vocabularies on training data; val/test reindexed to training columns.

In [6]:
PM_COLS        = ['case:concept:name', 'concept:name', 'time:timestamp']
SKIP_COLS      = {'case:concept:name', 'concept:name', 'time:timestamp', '@@diff_start_end'}
CASE_ATTR_COLS = ['Region', 'Country', 'Job Family', 'Job Family Group', 'CW', 'Evergreen']

case_attrs_train = train_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_val   = val_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_test  = test_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)


def get_prefix_pm(df_clean, k):
    if k == 'all':
        return df_clean[PM_COLS].copy()
    prefix = pm.get_prefixes_from_log(df_clean, length=k, case_id_key='case:concept:name')
    return prefix[PM_COLS].copy()


def bool_encoding(prefix_pm):
    enriched = pm.extract_outcome_enriched_dataframe(
        prefix_pm, activity_key='concept:name',
        timestamp_key='time:timestamp', case_id_key='case:concept:name',
    )
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return enriched.drop_duplicates('case:concept:name').set_index('case:concept:name')[feat_cols].sort_index()


def freq_encoding(prefix_pm):
    fea_df = pm.extract_features_dataframe(
        prefix_pm, activity_key='concept:name', timestamp_key='time:timestamp',
        case_id_key='case:concept:name', include_case_id=True, count_occurrences=True,
    )
    return fea_df.set_index('case:concept:name').sort_index()


def succ_encoding(prefix_pm):
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())
    df_s = prefix_pm.copy()
    df_s['prev_act'] = df_s.groupby('case:concept:name')['concept:name'].shift(1)
    df_s = df_s.dropna(subset=['prev_act'])
    if df_s.empty:
        return pd.DataFrame(index=pd.Index(all_case_ids, name='case:concept:name'))
    df_s['bigram'] = df_s['prev_act'] + ' >> ' + df_s['concept:name']
    bigram_counts = df_s.groupby(['case:concept:name', 'bigram']).size().unstack(fill_value=0)
    return bigram_counts.reindex(all_case_ids, fill_value=0).rename_axis('case:concept:name')


def build_Xy(X_pm, case_attrs_split, y_series, train_cols=None):
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    common = X.index.intersection(y_series.index)
    X = X.loc[common]
    y = y_series.reindex(common).values.astype(np.int64)
    return X.values.astype(np.float32), y, list(X.columns)


print('Encoding helpers defined.')

Encoding helpers defined.


## 6. Next-Activity Label Extraction

For prefix of length k, the target is the activity at position k+1 (0-indexed: `iloc[k]`).  
Cases shorter than k+1 events are excluded for that prefix length.

In [7]:
def extract_next_act_labels(df_subset, k, act2idx):
    """
    Return a Series: case_id -> next_activity_index.
    Excludes cases with <= k events (no next activity to predict).
    Unseen activities in val/test mapped to act2idx['<UNK>'].
    """
    records = {}
    for cid, grp in df_subset.groupby('case:concept:name', sort=False):
        grp = grp.sort_values('time:timestamp')
        if len(grp) <= k:
            continue
        next_act = grp.iloc[k]['concept:name']
        records[cid] = act2idx.get(next_act, act2idx['<UNK>'])
    return pd.Series(records, name='next_act')


# Also build padded sequences for DL (full prefix up to k)
def build_sequences(df_subset, case_ids, k, act2idx, max_seq_len):
    seqs = []
    df_s = df_subset.sort_values(['case:concept:name', 'time:timestamp'])
    grp_map = {cid: grp for cid, grp in df_s.groupby('case:concept:name', sort=False)}
    for cid in case_ids:
        grp = grp_map.get(cid)
        if grp is None:
            seqs.append([0] * max_seq_len)
            continue
        acts = grp['concept:name'].tolist()
        prefix_acts = acts[:k] if k != 'all' else acts
        idxs = [act2idx.get(a, act2idx['<UNK>']) for a in prefix_acts]
        seq  = idxs[-max_seq_len:]
        seqs.append([0] * (max_seq_len - len(seq)) + seq)
    return np.array(seqs, dtype=np.int64)


print('Label extraction functions defined.')

Label extraction functions defined.


## 7. Prefix-Length Sweep — LightGBM (Boolean × Frequency × Succession)

For each (encoding, k) pair: fit a LightGBM multiclass classifier with early stopping on the validation set.  
Metrics: Accuracy and Macro-F1.  
Prefix lengths: [1, 3, 5, 10, 20] — 'all' excluded because the full trace has no next activity.

In [ ]:
PREFIX_LENGTHS = [1, 3, 5, 10, 20]
ENCODINGS     = [('Boolean', bool_encoding), ('Frequency', freq_encoding), ('Succession', succ_encoding)]
sweep_results = {}  # (enc_name, k) -> {'Acc': float, 'F1': float, 'AP': float}

for enc_name, enc_fn in ENCODINGS:
    print(f'\n── {enc_name} encoding ──────────────────────')
    for k in PREFIX_LENGTHS:
        train_prefix_pm = get_prefix_pm(train_df, k)
        val_prefix_pm   = get_prefix_pm(val_df,   k)
        test_prefix_pm  = get_prefix_pm(test_df,  k)

        X_tr_pm    = enc_fn(train_prefix_pm)
        train_cols = list(X_tr_pm.columns)
        X_va_pm    = enc_fn(val_prefix_pm)
        X_te_pm    = enc_fn(test_prefix_pm)

        y_tr_s = extract_next_act_labels(train_df, k, act2idx)
        y_va_s = extract_next_act_labels(val_df,   k, act2idx)
        y_te_s = extract_next_act_labels(test_df,  k, act2idx)

        X_tr, y_tr, feat_names = build_Xy(X_tr_pm, case_attrs_train, y_tr_s)
        X_va, y_va, _          = build_Xy(X_va_pm, case_attrs_val,   y_va_s, train_cols=train_cols)
        X_te, y_te, _          = build_Xy(X_te_pm, case_attrs_test,  y_te_s, train_cols=train_cols)

        if len(X_tr) == 0 or len(X_te) == 0:
            print(f'  k={k}: skipping (empty split)')
            continue

        # Val may contain next-activity labels unseen in training at this prefix length.
        # Filter eval_set to known labels only (early stopping only; test is unaffected).
        train_label_set = set(y_tr.tolist())
        va_mask = np.isin(y_va, list(train_label_set))
        X_va_es, y_va_es = X_va[va_mask], y_va[va_mask]

        n_classes = int(y_tr.max()) + 1
        clf = LGBMClassifier(
            n_estimators=300, num_class=n_classes,
            class_weight='balanced', n_jobs=-1,
            random_state=RANDOM_STATE, verbose=-1
        )
        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_va_es, y_va_es)],
            callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)],
        )
        pred  = clf.predict(X_te)
        proba = clf.predict_proba(X_te)
        acc   = accuracy_score(y_te, pred)
        f1    = f1_score(y_te, pred, average='macro', zero_division=0)
        # Macro AUC-PR: one-vs-rest average precision over all activity classes
        ap    = average_precision_score(y_te, proba, average='macro')
        sweep_results[(enc_name, k)] = {'Acc': acc, 'F1': f1, 'AP': ap}
        print(f'  k={str(k):3s}  Acc={acc:.4f}  Macro-F1={f1:.4f}  AUC-PR={ap:.4f}')

## 8. Full-Trace Models (k = all)

RF / XGBoost / LightGBM trained on complete traces.  
Best encoding from sweep is used; all three are extracted for comparison.

In [ ]:
k_full = 20  # largest fixed prefix for full model comparison

y_tr_s_full = extract_next_act_labels(train_df, k_full, act2idx)
y_va_s_full = extract_next_act_labels(val_df,   k_full, act2idx)
y_te_s_full = extract_next_act_labels(test_df,  k_full, act2idx)

# Boolean
X_bool_tr_pm = bool_encoding(get_prefix_pm(train_df, k_full))
bool_cols    = list(X_bool_tr_pm.columns)
X_bool_va_pm = bool_encoding(get_prefix_pm(val_df,   k_full))
X_bool_te_pm = bool_encoding(get_prefix_pm(test_df,  k_full))
X_bool_tr, y_tr, bool_feat = build_Xy(X_bool_tr_pm, case_attrs_train, y_tr_s_full)
X_bool_va, y_va, _         = build_Xy(X_bool_va_pm, case_attrs_val,   y_va_s_full, bool_cols)
X_bool_te, y_te, _         = build_Xy(X_bool_te_pm, case_attrs_test,  y_te_s_full, bool_cols)

# Succession
X_succ_tr_pm = succ_encoding(get_prefix_pm(train_df, k_full))
succ_cols    = list(X_succ_tr_pm.columns)
X_succ_va_pm = succ_encoding(get_prefix_pm(val_df,   k_full))
X_succ_te_pm = succ_encoding(get_prefix_pm(test_df,  k_full))
X_succ_tr, _, succ_feat = build_Xy(X_succ_tr_pm, case_attrs_train, y_tr_s_full)
X_succ_va, _, _         = build_Xy(X_succ_va_pm, case_attrs_val,   y_va_s_full, succ_cols)
X_succ_te, _, _         = build_Xy(X_succ_te_pm, case_attrs_test,  y_te_s_full, succ_cols)

print(f'Boolean:    train={X_bool_tr.shape}, test={X_bool_te.shape}')
print(f'Succession: train={X_succ_tr.shape}, test={X_succ_te.shape}')

In [ ]:
full_results = {}
n_cls = int(y_tr.max()) + 1

def run_clf(X_train, X_val, X_test, y_train, y_val, y_test, label):
    models = {
        'RF':  RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
        'XGB': XGBClassifier(n_estimators=300, num_class=n_cls, objective='multi:softmax',
                             n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
                             early_stopping_rounds=20, eval_metric='mlogloss'),
        'LGBM':LGBMClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1,
                              random_state=RANDOM_STATE, verbose=-1),
    }
    res = {}
    for name, clf in models.items():
        print(f'  [{label}] {name}...')
        if name == 'XGB':
            clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        elif name == 'LGBM':
            clf.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)])
        else:
            clf.fit(X_train, y_train)
        pred  = clf.predict(X_test)
        proba = clf.predict_proba(X_test)
        all_classes = np.arange(proba.shape[1])
        acc   = accuracy_score(y_test, pred)
        f1    = f1_score(y_test, pred, average='macro', zero_division=0)
        top3  = top_k_accuracy_score(y_test, proba, k=3, labels=all_classes)
        ap    = average_precision_score(y_test, proba, average='macro')
        res[name] = {'Acc': acc, 'Macro-F1': f1, 'Top-3': top3, 'AUC-PR': ap}
        print(f'    Acc={acc:.4f}  Macro-F1={f1:.4f}  Top-3={top3:.4f}  AUC-PR={ap:.4f}')
    return res

print('=== Boolean Encoding (k=20) ===')
full_results['Boolean']    = run_clf(X_bool_tr, X_bool_va, X_bool_te, y_tr, y_va, y_te, 'Boolean')
print('\n=== Succession Encoding (k=20) ===')
full_results['Succession'] = run_clf(X_succ_tr, X_succ_va, X_succ_te, y_tr, y_va, y_te, 'Succession')

## 9. Deep Learning — LSTM + Transformer Classifier

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(torch.__version__)

k_dl = 20
y_tr_dl = extract_next_act_labels(train_df, k_dl, act2idx)
y_va_dl = extract_next_act_labels(val_df,   k_dl, act2idx)
y_te_dl = extract_next_act_labels(test_df,  k_dl, act2idx)

seqs_tr = build_sequences(train_df, y_tr_dl.index, k_dl, act2idx, MAX_SEQ_LEN)
seqs_va = build_sequences(val_df,   y_va_dl.index, k_dl, act2idx, MAX_SEQ_LEN)
seqs_te = build_sequences(test_df,  y_te_dl.index, k_dl, act2idx, MAX_SEQ_LEN)

# Tabular features (Boolean) for DL
X_tr_tab, _, _ = build_Xy(bool_encoding(get_prefix_pm(train_df, k_dl)), case_attrs_train, y_tr_dl)
X_va_tab, _, _ = build_Xy(bool_encoding(get_prefix_pm(val_df,   k_dl)), case_attrs_val,   y_va_dl, bool_cols)
X_te_tab, _, _ = build_Xy(bool_encoding(get_prefix_pm(test_df,  k_dl)), case_attrs_test,  y_te_dl, bool_cols)
N_TAB = X_tr_tab.shape[1]

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_tr_tab = scaler.fit_transform(X_tr_tab).astype(np.float32)
X_va_tab = scaler.transform(X_va_tab).astype(np.float32)
X_te_tab = scaler.transform(X_te_tab).astype(np.float32)


class NextActDataset(Dataset):
    def __init__(self, X, seqs, y):
        self.X    = torch.tensor(X,    dtype=torch.float32)
        self.seqs = torch.tensor(seqs, dtype=torch.long)
        self.y    = torch.tensor(y,    dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.seqs[i], self.y[i]

train_ds = NextActDataset(X_tr_tab, seqs_tr, y_tr_dl.values)
val_ds   = NextActDataset(X_va_tab, seqs_va, y_va_dl.values)
test_ds  = NextActDataset(X_te_tab, seqs_te, y_te_dl.values)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
class LSTMNextAct(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim=64, hidden=128, n_layers=2, dropout=0.3, n_tab=10):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc1  = nn.Linear(hidden + n_tab, 128)
        self.fc2  = nn.Linear(128, n_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        x = self.emb(seq)
        _, (h, _) = self.lstm(x)
        combined = torch.cat([h[-1], tab], dim=1)
        return self.fc2(torch.relu(self.fc1(self.drop(combined))))


def train_clf_model(model, train_loader, val_loader, n_epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_acc, best_state, no_improve = 0.0, None, 0

    for epoch in range(n_epochs):
        model.train()
        for X_b, seq_b, y_b in train_loader:
            X_b, seq_b, y_b = X_b.to(device), seq_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(model(seq_b, X_b), y_b).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds_val, targets_val = [], []
        with torch.no_grad():
            for X_b, seq_b, y_b in val_loader:
                X_b, seq_b = X_b.to(device), seq_b.to(device)
                preds_val.extend(model(seq_b, X_b).argmax(1).cpu().numpy())
                targets_val.extend(y_b.numpy())
        val_acc = accuracy_score(targets_val, preds_val)
        scheduler.step(-val_acc)
        print(f'  Epoch {epoch+1:02d}  val_Acc={val_acc:.4f}')

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5:
                print(f'  Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


lstm_model = LSTMNextAct(VOCAB_SIZE, VOCAB_SIZE, n_tab=N_TAB).to(device)
print('Training LSTM...')
lstm_model = train_clf_model(lstm_model, train_loader, val_loader)

lstm_model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        logits = lstm_model(seq_b, X_b)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_targets.extend(y_b.numpy())
all_probs = np.array(all_probs)
lstm_acc = accuracy_score(all_targets, all_preds)
lstm_f1  = f1_score(all_targets, all_preds, average='macro', zero_division=0)
lstm_ap  = average_precision_score(all_targets, all_probs, average='macro')
print(f'LSTM  Acc={lstm_acc:.4f}  Macro-F1={lstm_f1:.4f}  AUC-PR={lstm_ap:.4f}')

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerNextAct(nn.Module):
    def __init__(self, vocab_size, n_classes, emb_dim=64, nhead=4, n_layers=2,
                 d_ff=128, dropout=0.1, n_tab=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.pe  = PositionalEncoding(emb_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                                               dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.fc1 = nn.Linear(emb_dim + n_tab, 128)
        self.fc2 = nn.Linear(128, n_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, seq, tab):
        pad_mask = (seq == 0)
        x = self.pe(self.emb(seq))
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).unsqueeze(-1).float()
        x = (x * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([x, tab], dim=1)))))


tf_model = TransformerNextAct(VOCAB_SIZE, VOCAB_SIZE, n_tab=N_TAB).to(device)
print('Training Transformer...')
tf_model = train_clf_model(tf_model, train_loader, val_loader)

tf_model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        X_b, seq_b = X_b.to(device), seq_b.to(device)
        logits = tf_model(seq_b, X_b)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_targets.extend(y_b.numpy())
all_probs = np.array(all_probs)
tf_acc = accuracy_score(all_targets, all_preds)
tf_f1  = f1_score(all_targets, all_preds, average='macro', zero_division=0)
tf_ap  = average_precision_score(all_targets, all_probs, average='macro')
print(f'Transformer  Acc={tf_acc:.4f}  Macro-F1={tf_f1:.4f}  AUC-PR={tf_ap:.4f}')

## 10. Results Table + Accuracy vs Prefix Plot

In [ ]:
print('\n=== SWEEP RESULTS ===')
for (enc, k), m in sorted(sweep_results.items(), key=lambda x: (x[0][0], x[0][1])):
    print(f'{enc:12s}  k={str(k):3s}  Acc={m["Acc"]:.4f}  Macro-F1={m["F1"]:.4f}  AUC-PR={m["AP"]:.4f}')

print('\n=== FULL-TRACE MODELS (k=20) ===')
for enc, res in full_results.items():
    for name, m in res.items():
        print(f'{enc:12s}  {name:6s}  Acc={m["Acc"]:.4f}  Macro-F1={m["Macro-F1"]:.4f}  '
              f'Top-3={m["Top-3"]:.4f}  AUC-PR={m["AUC-PR"]:.4f}')

print(f'\nLSTM         Acc={lstm_acc:.4f}  Macro-F1={lstm_f1:.4f}  AUC-PR={lstm_ap:.4f}')
print(f'Transformer  Acc={tf_acc:.4f}  Macro-F1={tf_f1:.4f}  AUC-PR={tf_ap:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_keys = [('Acc', 'Accuracy'), ('F1', 'Macro-F1'), ('AP', 'AUC-PR (macro)')]
for ax, (mk, title) in zip(axes, metric_keys):
    for enc_name, _ in ENCODINGS:
        vals = [sweep_results.get((enc_name, k), {}).get(mk, np.nan) for k in PREFIX_LENGTHS]
        ax.plot([str(k) for k in PREFIX_LENGTHS], vals, marker='o', label=enc_name)
    ax.set_title(f'Next Activity: {title} vs Prefix Length')
    ax.set_xlabel('Prefix Length'); ax.set_ylabel(title); ax.legend()
plt.tight_layout()
plt.savefig('figs/next_act_metrics_vs_prefix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/next_act_metrics_vs_prefix.png')